# A language model made of nothing but counts

*A companion to the blog post “A language model made of nothing but counts” (Part 5 of the Tokenization & decoding series).*

You will build a character n-gram model, measure its perplexity on text it has not seen, fix its failures with smoothing, and sample text from it. Runs in seconds on the free Colab CPU.

In [ ]:
import collections, math, random

TEXT = 'The warehouse opens at six in the morning and the first trucks arrive soon after. Workers check each pallet against the shipping list and mark any box that looks damaged. When the list is complete the driver signs the form and the truck leaves for the next stop. Every shipment has a number, and every number tells the team where the parcel has been. If a parcel is late, the team looks at the last scan and calls the carrier to ask why. Customers want to know when their order will arrive, so the system sends a message each time the parcel moves from one place to another. A good message is short and clear. It tells the customer what happened, where the parcel is now, and when it should arrive. Most days nothing goes wrong, and the trucks leave on time. On the other days the team works together to find the problem and fix it before the end of the shift. In the afternoon the second wave of trucks comes in from the port. The drivers are tired, but they still wait in line, and the workers still check every pallet with care. A pallet that is counted twice causes a mistake in the stock list, and a pallet that is not counted at all causes a bigger one. For this reason two people look at the same list at the same time. When the numbers agree they sign, and when they do not agree they count again from the start. The office on the first floor keeps a record of every call that comes in. Most calls are simple: a customer asks where the order is, and the answer is on the screen. Some calls are harder, for example when a parcel has been sent to the wrong city, or when the address on the label cannot be read. In those cases the team writes a short note, and the note is sent to the carrier together with a photo of the label. The carrier replies within a day, and the customer receives a new date for delivery. Over the years the team has learned that clear words matter more than long words. A message that says the parcel will arrive on Friday is better than a message that says the delivery window has been adjusted. People read the first line and decide whether to worry. At night the building is quiet. The machines stop, the lights go down, and only the night guard walks between the shelves. In the morning the cycle begins again, with the same trucks, the same lists, and the same careful counting, because a small habit repeated every day is what keeps a large system running. '
cut = int(len(TEXT) * 0.9)
train, held = TEXT[:cut], TEXT[cut:]          # train on 90%, test on the last 10%
V = len(set(TEXT))                           # number of different symbols
print(len(TEXT), 'characters,', V, 'symbols; guessing at random gives perplexity', V)

## 1. Count what follows each context
`models[n][context][next_letter]` = how often `next_letter` followed the previous `n-1` letters.

In [ ]:
models = {}
for n in range(1, 8):
    cnt = collections.defaultdict(collections.Counter)
    for i in range(n - 1, len(train)):
        cnt[train[i - n + 1:i]][train[i]] += 1
    models[n] = cnt

print('after "th":', models[3]['th'].most_common(5))

## 2. Three ways to turn counts into probabilities

In [ ]:
def p_mle(n, ctx, ch):                      # raw counts
    c = models[n].get(ctx)
    return c[ch] / sum(c.values()) if c and ch in c else 0.0

def p_addk(n, ctx, ch, k=0.1):              # add a little to every count
    c = models[n].get(ctx, collections.Counter())
    return (c[ch] + k) / (sum(c.values()) + k * V)

def p_interp(n, ctx, ch, disc=0.75):        # back off to a shorter context when this one is rare
    if n == 1:
        return (models[1][''][ch] + 0.5) / (sum(models[1][''].values()) + 0.5 * V)
    c = models[n].get(ctx); lower = p_interp(n - 1, ctx[1:], ch)
    if not c:
        return lower
    tot = sum(c.values())
    return max(c[ch] - disc, 0) / tot + (disc * len(c) / tot) * lower

## 3. Perplexity on text the model has not seen
Perplexity = e raised to the average negative log-probability per letter. A probability of exactly 0 is clipped to 1e-12 so the number stays finite.

In [ ]:
def perplexity(prob, n):
    ll = m = 0
    for i in range(n - 1, len(held)):
        ll += math.log(max(prob(n, held[i - n + 1:i], held[i]), 1e-12)); m += 1
    return math.exp(-ll / m)

print('letters of context | raw counts | add-0.1 | backing off')
for n in range(1, 7):
    print(f'{n - 1:>18} | {perplexity(p_mle, n):>10,.1f} | {perplexity(p_addk, n):>7.1f} | {perplexity(p_interp, n):>7.1f}')

**What to notice.** Raw counts get *worse* as the context grows, because long contexts in the test text were never seen in training and get probability 0. Backing off keeps improving until about 3 letters of context. Random guessing would score 40.

## 4. Let the model write
Same text, same random seed, three context lengths:

In [ ]:
def generate(n, length=150, seed=1):
    r = random.Random(seed); out = train[:n - 1]
    for _ in range(length):
        c = models[n].get(out[-(n - 1):] if n > 1 else '')
        if not c:
            break
        ch, w = zip(*c.items()); out += r.choices(ch, w)[0]
    return out

for n in (2, 4, 6):
    print(f'{n - 1} letters of context:', repr(generate(n)[:110]), '\n')

**Try it:** paste a paragraph of your own into `TEXT` (at least 2,000 characters) and re-run. Which context length is best for your text?

### Want the full guided version?

This notebook is a small taster from **Chapter 2 of the course *Generative AI and LLMs with Python*** (course-2-generative-ai), one of 4 world-class courses on [aibits.ecubeanalytics.com](https://aibits.ecubeanalytics.com/?utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch02).

**The full guided lab, quizzes and projects are in the course.**

- [Open Chapter 2 in the course (free sign-up where the chapter is in the free first module)](https://aibits.ecubeanalytics.com/main/document/genai_gate.php?c=course-2-generative-ai&f=chapters/course-2-generative-ai/02-tokenization-lm-objective-decoding.html&utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch02)
- [See the course demo](https://aibits.ecubeanalytics.com/main/document/genai_public/GENAILLMPYTHON.html?utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch02)
